# 🧭 Unity Catalog — Knowledge Transfer Notebook

### A beginner-friendly, hands-on introduction to Databricks Unity Catalog

Welcome! 👋

This notebook is designed for engineers who are **completely new to Databricks and Unity Catalog**.

You don't need any prior Unity Catalog knowledge to follow along. We'll build up the concepts
step by step, using one running example: **Microsoft** and how it manages its data.

By the end of this notebook, you should feel comfortable:
- Explaining what Unity Catalog is and why it exists
- Navigating the Unity Catalog hierarchy (Metastore → Catalog → Schema → Table)
- Running basic SQL and PySpark commands against Unity Catalog
- Understanding what happens "under the hood" when a query runs

Let's get started 🚀

## 🎯 Learning Objectives

By the end of this session, you will be able to:

1. Explain **why** companies need Unity Catalog
2. Describe the difference between **data** and **metadata**
3. Draw and explain the **Unity Catalog hierarchy**
4. Run basic SQL commands: `CREATE CATALOG`, `CREATE SCHEMA`, `CREATE TABLE`, etc.
5. Query `INFORMATION_SCHEMA` to explore metadata
6. Describe what happens internally when a query runs against a governed table
7. Recognize Unity Catalog objects like **Volumes**, **External Locations**, **Storage Credentials**
8. Understand **Row Filters** and **Column Masking** at a high level

> 💡 **Note:** This notebook is for KT / onboarding purposes — it is **not** a certification prep guide.

## 🤔 Before We Start... Ask Yourself One Question

> **"When a company stores 100 TB of data, how do thousands of employees access it securely?"**

Keep that question in mind. Everything in this notebook is really just one long answer to it.

Let's make it concrete with an example.

## 🏢 Our Running Example: Microsoft

Let's imagine **Microsoft**. Like any large company, they have data spread across many teams:

| Team / Type      | Example Data                     |
|-------------------|-----------------------------------|
| 👩‍💼 HR            | Employee records, salaries        |
| 💰 Finance         | Budgets, revenue reports          |
| 🧑‍🤝‍🧑 Customer Data | Customer profiles, support tickets|
| 📈 Sales           | Deals, pipelines, targets         |
| 📣 Marketing       | Campaigns, ad performance         |
| 🤖 AI Models       | Trained ML models                 |
| 🖼️ Images          | Product photos, design assets     |
| 📄 PDFs            | Contracts, reports                |

Everything is stored **somewhere**.

**Question:** How do they manage all of this — securely, and at scale?

We'll answer this by first imagining what life looks like **without** Unity Catalog.

## 😱 Imagine There Is NO Unity Catalog

Suppose Microsoft stores everything directly in **Amazon S3**, organized into folders:

```
s3://company-data/
├── sales/
├── finance/
├── hr/
├── marketing/
└── customers/
```

Looks simple, right? It works fine... until **5000 employees join**.

Then the questions start coming in:

- ❓ Can HR access Finance?
- ❓ Can Sales modify HR data?
- ❓ Can interns delete production tables?
- ❓ Can one team see another team's salary data?
- ❓ Can we know **who deleted a table yesterday**?
- ❓ Can 3 Databricks workspaces share the same data?

Now imagine answering **all** of these questions **manually**, folder by folder, team by team.

**It's impossible to manage this at scale with folders alone.**

## 📌 The Real Problem Statement

Companies don't just need **storage**. They need:

- 🔐 **Governance**
- 🛡️ **Security**
- 🗂️ **Metadata**
- ✅ **Permissions**
- 📝 **Auditing**
- 🔍 **Discovery**
- 🧬 **Lineage**
- 🤝 **Sharing**

**This is exactly where Unity Catalog comes in.**

## 🧠 So... What Is Unity Catalog?

**Official definition:**

> "Unity Catalog is Databricks' unified governance solution for data and AI."

That sounds a bit formal for a first week on the job, so here's the simple version:

> 🧠 **Unity Catalog is the brain of your data platform.**

Unity Catalog tells Databricks:

- 📦 What data exists
- 📍 Where it is stored
- 👤 Who can access it
- ✅ What they can do with it
- 📝 How to track it
- 🔒 How to secure it

Now notice something important...

## ⚠️ Unity Catalog Does NOT Store Data

This is the single most important idea in this entire notebook. Read it twice. 😄

Suppose you have a table called `sales.delta`. Where does it physically live?

**Not inside Unity Catalog.**

It actually lives inside your cloud storage:

- ☁️ AWS S3, **or**
- ☁️ Azure ADLS, **or**
- ☁️ Google Cloud Storage

Unity Catalog only stores **information about** the data — things like:

- Table Name
- Owner
- Location
- Permissions
- Schema
- Columns
- Tags
- Lineage

**This "information about the data" is called `Metadata`.**

## 📦 Data vs Metadata — A Simple Way to Remember It

```
 ┌────────────────────────────┐        ┌──────────────────────────────┐
 │        Cloud Storage        │        │         Unity Catalog         │
 │   (S3 / ADLS / GCS)         │        │        (Metastore)            │
 │                              │        │                                │
 │   The actual Delta files     │        │   Table name, owner, columns  │
 │   (the DATA itself)          │        │   location, permissions, tags │
 │                              │        │            (METADATA)         │
 └────────────────────────────┘        └──────────────────────────────┘
```

**Key takeaway:**

> 🏬 **Storage stores the data. Unity Catalog governs the data.**

## 🕰️ Why Did Databricks Create Unity Catalog?

Before Unity Catalog, **every Databricks workspace had its own Hive Metastore**.

```
 Workspace A  →  Hive Metastore A
 Workspace B  →  Hive Metastore B
 Workspace C  →  Hive Metastore C
```

Each workspace had:

- ❌ Different permissions
- ❌ Different owners
- ❌ Different metadata

In short: **chaos**. The same table could mean different things — or have different access
rules — depending on which workspace you happened to be using.

## ✅ With Unity Catalog: One Source of Truth

```
        Workspace A
              │
              ▼
Workspace B ──▶  Unity Catalog  ◀── Workspace C
              ▲
              │
        (shared governance)
```

Now **every workspace shares the same governance layer**.

- One place for permissions
- One place for ownership
- One place for auditing

> 🎯 **One source of truth** for the entire company — no matter how many workspaces exist.

## 🛠️ What Problems Does Unity Catalog Solve?

Unity Catalog centralizes everything related to **governance**:

| Category | Examples |
|---|---|
| **Identity** | Users, Groups, Service Principals |
| **Data Objects** | Catalogs, Schemas, Tables, Views, Volumes, Functions, Models |
| **Governance** | Permissions, Policies, Row Filters, Column Masks |
| **Operational** | Lineage, Audit Logs, Credentials, External Locations |
| **Organization** | Tags, Comments, Metadata |

Storage vendors (S3, ADLS, GCS) never went away — Unity Catalog just sits **on top of them**
and governs how everyone interacts with what's inside.

## 🏗️ The Unity Catalog Hierarchy

Unity Catalog organizes everything in a simple, four-level hierarchy:

```
        Metastore
            │
         Catalog
            │
         Schema
            │
          Table
```

Let's map this to our Microsoft example:

```
        Metastore (Microsoft's single, company-wide governance layer)
            │
   ┌────────┼─────────┐
 sales   finance      hr        ← Catalogs (business domains)
   │
 gold, silver, bronze            ← Schemas (sub-groupings within a catalog)
   │
 customers, orders, leads        ← Tables (the actual structured data)
```

Let's look at each level one at a time.

### 1️⃣ Metastore

> A **Metastore** is the **top-level metadata repository** in Unity Catalog. It stores
> information about all data objects (catalogs, schemas, tables, permissions, owners, etc.).
>
> It stores **metadata only** — never the actual data.

Think of it as the single "phone book" for an entire company's data estate. Normally, an
organization has **one Metastore per region**, shared across all its workspaces.

### 2️⃣ Catalog

> A **Catalog** is the **highest level of logical organization** inside a Metastore. It groups
> related data assets, typically based on **business domains** such as Finance, HR, or Sales.

For Microsoft, this might look like:

```
Metastore
 ├── sales_catalog
 ├── finance_catalog
 └── hr_catalog
```

### 3️⃣ Schema

> A **Schema** is a container within a Catalog that organizes related database objects such as
> tables, views, functions, and volumes.

Inside `sales_catalog`, Microsoft might have:

```
sales_catalog
 ├── bronze   (raw sales data)
 ├── silver   (cleaned sales data)
 └── gold     (curated, reporting-ready sales data)
```

### 4️⃣ Table

> A **Table** is a structured data object that stores data in rows and columns. Unity Catalog
> supports both **Managed Tables** and **External Tables**.

Inside `sales_catalog.gold`, Microsoft might have:

```
sales_catalog.gold.customers
sales_catalog.gold.orders
sales_catalog.gold.leads
```

So the **fully qualified name** of a table always follows this pattern:

```
catalog.schema.table
```

Example: `sales_catalog.gold.customers`

## 🧪 Hands-On Lab: Let's Use Unity Catalog

Time to stop reading and start typing! 💻

In this section, we'll walk through the Unity Catalog hierarchy **hands-on**, using SQL and
PySpark. Run each cell in order, and read the explanation that follows.

> ⚠️ You need `CREATE CATALOG` privileges on your Metastore to run the `CREATE CATALOG`
> examples below. If you don't have that privilege, follow along by reading the outputs
> your instructor shares during the KT session.

In [0]:
%sql
-- Let's see what catalogs already exist in our Metastore
SHOW CATALOGS;

**What just happened?**

`SHOW CATALOGS` asks Unity Catalog: *"What catalogs exist that I'm allowed to see?"*

You'll typically see built-in catalogs like `main`, `system`, and `samples`, plus any
catalogs your organization has already created.

In [0]:
# The PySpark equivalent of SHOW CATALOGS
spark.sql("SHOW CATALOGS").display()

### 📝 Exercise 1

Run `SHOW CATALOGS` and note down how many catalogs you can see. Can you spot the `main`
catalog?

**Solution:** `SHOW CATALOGS;` — the `main` catalog is created by default in most workspaces.

In [0]:
%sql
-- Let's create our own catalog, following the Microsoft example
CREATE CATALOG IF NOT EXISTS sales_catalog
COMMENT 'Catalog for all Microsoft Sales data';

**What just happened?**

We created a brand-new **Catalog** called `sales_catalog`. Remember — this doesn't create any
storage or move any files. It creates an **entry in the Unity Catalog metastore** saying
"this catalog exists, and here's who owns it."

In [0]:
%sql
-- Switch our session to use this catalog by default
USE CATALOG sales_catalog;

**What just happened?**

`USE CATALOG` sets the **default catalog** for the current session, so we don't have to type
`sales_catalog.` in front of every command.

In [0]:
%sql
-- See what schemas exist inside sales_catalog
SHOW SCHEMAS;

**What just happened?**

Right now, `sales_catalog` likely only has the built-in `information_schema` and `default`
schemas. Let's create our own.

In [0]:
%sql
-- Create a schema for curated, reporting-ready sales data
CREATE SCHEMA IF NOT EXISTS sales_catalog.gold
COMMENT 'Curated sales data, ready for reporting';

### 📝 Exercise 2

Create one more schema called `silver` inside `sales_catalog`, for "cleaned but not yet
curated" data.

**Solution:**
```sql
CREATE SCHEMA IF NOT EXISTS sales_catalog.silver
COMMENT 'Cleaned sales data';
```

In [0]:
%sql
-- Now let's create a table inside our new schema
CREATE TABLE IF NOT EXISTS sales_catalog.gold.customers (
  customer_id INT,
  customer_name STRING,
  region STRING,
  total_purchase_usd DOUBLE
)
COMMENT 'Customer master data for Sales';

**What just happened?**

We created a **Managed Table**. Unity Catalog automatically decided where the underlying
Delta files should live (in a Databricks-managed storage location), and it recorded the
table's schema, owner, and location as **metadata**.

In [0]:
%sql
-- Let's insert a few sample rows
INSERT INTO sales_catalog.gold.customers VALUES
  (1, 'Contoso Ltd', 'US', 125000.50),
  (2, 'Fabrikam Inc', 'EU', 87250.00),
  (3, 'Northwind Traders', 'APAC', 45230.75);

In [0]:
%sql
-- Query the table
SELECT * FROM sales_catalog.gold.customers;

**What just happened?**

Nothing fancy on the surface — a simple `SELECT`. But behind the scenes, Unity Catalog
checked **who you are** and **what you're allowed to see** before Spark ever touched a single
Delta file. We'll unpack this "behind the scenes" flow in detail a bit later.

In [0]:
# The same query, in PySpark
df = spark.sql("SELECT * FROM sales_catalog.gold.customers")
df.display()

In [0]:
%sql
-- List all tables inside a schema
SHOW TABLES IN sales_catalog.gold;

In [0]:
%sql
-- Look at the table's column definitions
DESCRIBE TABLE sales_catalog.gold.customers;

In [0]:
%sql
-- Look at deeper details: location, format, size, etc.
DESCRIBE DETAIL sales_catalog.gold.customers;

**What just happened?**

`DESCRIBE TABLE` shows you the **schema** (column names and types).

`DESCRIBE DETAIL` shows you the **full metadata record** Unity Catalog is holding for this
table — including its physical storage `location`, format, size, and more. This is a great
way to *prove to yourself* that the data lives in cloud storage, not "inside" Unity Catalog.

In [0]:
%sql
-- What catalog and schema is my session currently using?
SELECT current_catalog(), current_schema();

In [0]:
%sql
-- Alternative syntax for the same thing
SHOW CURRENT CATALOG;
SHOW CURRENT SCHEMA;

In [0]:
%sql
-- Which Metastore is this workspace attached to?
SELECT current_metastore();

**What just happened?**

These commands are useful for **debugging** — especially when you're working across multiple
catalogs and want a quick sanity check on "where am I right now?"

### 📝 Exercise 3

Switch to a different catalog (e.g. `main`) using `USE CATALOG main;`, then run
`SELECT current_catalog();` again. What do you expect to see?

**Solution:** It will now return `main`, confirming the session default has changed.

## 🔍 INFORMATION_SCHEMA — Querying Metadata with SQL

We've been using commands like `SHOW TABLES` and `DESCRIBE TABLE` to peek at metadata. But
Unity Catalog also gives us a much more powerful tool: **`INFORMATION_SCHEMA`**.

> **`INFORMATION_SCHEMA`** is a **read-only schema** that provides SQL access to metadata
> about database objects. It does **not** store the metadata itself — it provides a way to
> **query** the metadata that the Metastore already holds.

```
      Unity Catalog Metastore
       (Actual Metadata Storage)
                 │
                 │
        INFORMATION_SCHEMA
      (Read-only SQL Interface)
                 │
                 │
           Your SQL Query
```

So remember:

- **Metastore** = actually **stores** metadata
- **INFORMATION_SCHEMA** = lets you **read** that metadata using plain SQL

In [0]:
%sql
-- List every column across every table you can see in this catalog
SELECT *
FROM sales_catalog.INFORMATION_SCHEMA.COLUMNS;

**What just happened?**

This query didn't touch any actual data — it only read **metadata** describing columns:
table name, column name, data type, position, and more. Extremely useful for building
data catalogs, documentation, or automated audits.

In [0]:
%sql
-- See who has been granted what privileges on each table
SELECT *
FROM sales_catalog.INFORMATION_SCHEMA.TABLE_PRIVILEGES;

In [0]:
%sql
-- See constraints (primary keys, foreign keys, etc.) defined on tables
SELECT *
FROM sales_catalog.INFORMATION_SCHEMA.TABLE_CONSTRAINTS;

In [0]:
%sql
-- Find every EXTERNAL table in this catalog
SELECT *
FROM sales_catalog.INFORMATION_SCHEMA.TABLES
WHERE table_type = 'EXTERNAL';

### 📝 Exercise 4

Write a query against `INFORMATION_SCHEMA.TABLES` that returns only tables owned by you.

**Solution:**
```sql
SELECT table_catalog, table_schema, table_name, table_owner
FROM sales_catalog.INFORMATION_SCHEMA.TABLES
WHERE table_owner = current_user();
```

## ⚙️ How Does Unity Catalog Work Internally?

This is usually the "aha!" moment for new engineers. Let's walk through it carefully.

Imagine you execute:

```sql
SELECT * FROM sales_catalog.gold.customers;
```

You might assume Spark immediately jumps in and reads the table. **That's wrong.** A lot
happens before a single row is returned. Let's trace it step by step.

### The Journey of a Query

```
 You
  │  SELECT * FROM sales_catalog.gold.customers;
  ▼
 Spark
  │  "Does this table exist?"
  ▼
 Unity Catalog  ──▶  checks its metadata
  │
  ├── Not found?  ───────────────▶  ❌ Error
  │
  ▼  Found!
 "Who are you?"  ──▶  e.g. Gokul
  │
  ▼
 "Do you have SELECT permission?"
  │
  ├── No  ───────────────────────▶  ❌ Permission denied
  │
  ▼  Yes
 "Any row filters?"  ──▶  e.g. department = 'HR'
  │        (Unity Catalog silently adds
  │         WHERE department = 'HR' to your query)
  ▼
 "Any column masks?"  ──▶  e.g. salary column → ******
  │
  ▼
 "Where is the table physically stored?"
  │  ──▶  s3://company-data/gold/customers
  ▼
 "Which credential should I use to access that path?"
  │  ──▶  IAM Role
  ▼
 Spark reads the Delta files
  │
  ▼
 ✅ Results are returned to you
```

### Step-by-Step Breakdown

| Step | What Happens |
|---|---|
| 1 | You submit the query `SELECT * FROM sales.customers;` |
| 2 | Spark asks Unity Catalog: *does this table exist?* Unity Catalog checks its metadata — if not found, you get an error. |
| 3 | Unity Catalog checks: *who are you?* (your identity) |
| 4 | Unity Catalog checks: *do you have `SELECT` permission?* If not, permission denied. |
| 5 | Unity Catalog checks for **row filters** (e.g. `department = 'HR'`) and silently adds them to your query. |
| 6 | Unity Catalog checks for **column masks** — e.g. a `salary` column might become `******`. |
| 7 | Unity Catalog looks up **where the table is physically stored**, e.g. `s3://company-data/gold/customer`. |
| 8 | Unity Catalog determines **which credential** to use to access that storage, e.g. an IAM Role. |
| 9 | Spark finally reads the actual Delta files. |
| 10 | Results are returned to you. |

⏱️ **This entire process takes milliseconds** — but it shows exactly why Unity Catalog sits
in the **critical path** for governance on every single query.

## 🗄️ So... Where Exactly Is the Metadata Stored?

Let's ground this with our `customers` table example.

**First — where is the actual data?**

When you run `CREATE TABLE ...`, the actual Delta files are stored in your cloud storage
(AWS S3, Azure ADLS, or GCP GCS).

**For that table, Unity Catalog stores metadata such as:**

- Table Name — `customers`
- Catalog — `sales_catalog`
- Schema — `gold`
- Owner — `Gokul`
- Columns — `customer_id`, `customer_name`, ...
- Location — `s3://company-data/gold/customers`
- Created Time
- Permissions
- Comments
- Tags

**All of this metadata is stored inside the Unity Catalog Metastore** — not as regular Delta
tables you can casually browse, but in Databricks' own internal, managed metadata store.

### What Exactly *Is* a Metastore?

A common point of confusion for beginners: **a Metastore is not a Catalog.**

> A **Metastore** is a **metadata database** managed by Databricks.

In one sentence:

```
Unity Catalog  =>  Metastore  =>  Stores Metadata
```

Every Databricks account typically has **one Metastore per region**, and every workspace in
that region attaches to it — which is exactly what gives us the "one source of truth" we
talked about earlier.

## 🧩 Quick Introduction to Other Unity Catalog Objects

So far we've focused on the core hierarchy (Metastore → Catalog → Schema → Table). Let's
briefly meet a few other important Unity Catalog objects you'll run into on real projects.

### 📁 Volume

> A **Volume** is a governed storage object used to store and manage **unstructured files**
> such as PDFs, images, CSVs, Excel files, JSON files, machine learning models, and other
> documents.

Remember Microsoft's **Images, PDFs, and AI Models** from our very first example? Volumes
are exactly how Unity Catalog governs access to files like these — the same way tables
govern access to structured rows and columns.

In [0]:
%sql
-- Create a Volume to hold Sales contract PDFs
CREATE VOLUME IF NOT EXISTS sales_catalog.gold.contracts_volume
COMMENT 'Governed storage for Sales contract PDFs and scanned documents';

**What just happened?**

Just like `CREATE TABLE`, this only creates a **metadata entry** for the volume. Unity Catalog
now knows this volume exists, who owns it, and — once files are added — where they live.

Every volume gets a predictable path of the form:

```
/Volumes/<catalog>/<schema>/<volume_name>/
```

In [0]:
%sql
-- List files currently inside the volume (empty for now)
LIST '/Volumes/sales_catalog/gold/contracts_volume';

In [0]:
# Copying a file into the volume using dbutils (PySpark / notebook context)
dbutils.fs.cp(
    "file:/tmp/contoso_msa_2026.pdf",
    "/Volumes/sales_catalog/gold/contracts_volume/contoso_msa_2026.pdf"
)

# Now list the volume again to confirm the file landed
display(dbutils.fs.ls("/Volumes/sales_catalog/gold/contracts_volume"))

### 📝 Exercise 5

Create a second volume called `product_images_volume` inside `sales_catalog.gold` for
storing product photos.

**Solution:**
```sql
CREATE VOLUME IF NOT EXISTS sales_catalog.gold.product_images_volume
COMMENT 'Governed storage for product images';
```

### 🔑 Storage Credential

> A **Storage Credential** is a secure object that stores **cloud authentication details**
> (such as an AWS IAM Role, Azure Managed Identity, or GCP Service Account) used by Unity
> Catalog to access cloud storage **without exposing credentials to users**.

This is why, back in our "how does a query run internally" walkthrough, Unity Catalog could
say *"use this IAM Role"* — without you ever seeing or handling the actual cloud secret.

> ⚠️ Creating a Storage Credential requires **Metastore Admin** privileges — this is normally
> a one-time setup done by a platform/admin team, not by every engineer. It's shown here so
> you understand what it looks like when you come across one.

In [0]:
%sql
-- Azure example: register a managed identity as a Storage Credential
CREATE STORAGE CREDENTIAL IF NOT EXISTS microsoft_sales_credential
AZURE_MANAGED_IDENTITY
  ACCESS_CONNECTOR_ID 'subscriptions/<sub-id>/resourceGroups/<rg-name>/providers/Microsoft.Databricks/accessConnectors/<connector-name>'
COMMENT 'Managed identity used to access Microsoft Sales data in ADLS Gen2';

In [0]:
%sql
-- AWS equivalent, for reference: register an IAM Role as a Storage Credential
-- CREATE STORAGE CREDENTIAL IF NOT EXISTS microsoft_sales_credential
-- AWS_IAM_ROLE
--   ROLE_ARN 'arn:aws:iam::123456789012:role/unity-catalog-sales-role'
-- COMMENT 'IAM role used to access Microsoft Sales data in S3';

In [0]:
%sql
-- Confirm the credential was created, and inspect its details
SHOW STORAGE CREDENTIALS;
DESCRIBE STORAGE CREDENTIAL microsoft_sales_credential;

**What just happened?**

Unity Catalog stored a **reference** to a cloud identity (an Azure Managed Identity here).
Nobody querying data ever sees the underlying secret — Unity Catalog exchanges it for
temporary, short-lived cloud access on their behalf.

### 🌐 External Location

> An **External Location** is a Unity Catalog object that securely maps a **cloud storage
> path** (S3, ADLS, or GCS) to a **Storage Credential**, enabling governed access to external
> data.

Think of it as: *"This particular storage folder is allowed to be accessed, and here's the
credential to use to get there."* An External Location always points at **one** Storage
Credential — which is why we created the credential first.

In [0]:
%sql
-- Map an ADLS path to the Storage Credential we just created
CREATE EXTERNAL LOCATION IF NOT EXISTS sales_external_location
URL 'abfss://sales@microsoftdatalake.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL microsoft_sales_credential)
COMMENT 'External location pointing to raw Sales data in ADLS';

In [0]:
%sql
-- Confirm it, and see which credential it's tied to
SHOW EXTERNAL LOCATIONS;
DESCRIBE EXTERNAL LOCATION sales_external_location;

In [0]:
%sql
-- Now we can create an EXTERNAL table on top of that governed path
CREATE TABLE IF NOT EXISTS sales_catalog.gold.customers_external (
  customer_id INT,
  customer_name STRING,
  region STRING
)
LOCATION 'abfss://sales@microsoftdatalake.dfs.core.windows.net/external/customers/';

**What just happened?**

Unlike our earlier **Managed Table** (where Unity Catalog chose the storage location for
us), this is an **External Table** — the data lives at a path *we* specify, but Unity Catalog
still governs who can query it, because that path sits inside a registered External Location.

### ✅ Permissions

> **Permissions** are access control rules that determine which **users, groups, or service
> principals** can perform actions (such as `SELECT`, `MODIFY`, `CREATE`, or `DELETE`) on
> Unity Catalog objects.

In [0]:
%sql
-- Grant the Finance team read access to the Sales customers table
GRANT SELECT ON TABLE sales_catalog.gold.customers TO `finance_team`;

-- Permissions cascade downward, so teams also need USE privileges "on the way down"
GRANT USE CATALOG ON CATALOG sales_catalog TO `finance_team`;
GRANT USE SCHEMA ON SCHEMA sales_catalog.gold TO `finance_team`;

In [0]:
%sql
-- Check exactly what's been granted on the table
SHOW GRANTS ON TABLE sales_catalog.gold.customers;

In [0]:
%sql
-- Removing access is just as simple
REVOKE SELECT ON TABLE sales_catalog.gold.customers FROM `finance_team`;

**What just happened?**

`GRANT` and `REVOKE` are how you actually answer the "Can Sales modify HR data?" style
questions from the very start of this notebook — explicitly, auditably, and without touching
a single storage folder permission.

### 📝 Exercise 6

Grant `SELECT` on `sales_catalog.gold.customers` to a group called `marketing_team`, then
confirm it with `SHOW GRANTS`.

**Solution:**
```sql
GRANT SELECT ON TABLE sales_catalog.gold.customers TO `marketing_team`;
SHOW GRANTS ON TABLE sales_catalog.gold.customers;
```

### 🚧 Row Filters (Row-Level Security)

> **Row-Level Security (Row Filters)** is a security feature that restricts **which rows** of
> a table a user can view, based on predefined access rules.

This is exactly the `WHERE department = 'HR'` example from our internal-workings walkthrough
— except Unity Catalog applies it **automatically**, without the user writing it.

A row filter is really just a **SQL function** that returns `TRUE` (show the row) or `FALSE`
(hide the row), attached to a table.

In [0]:
%sql
-- Step 1: write a function that decides which rows a user can see
CREATE OR REPLACE FUNCTION sales_catalog.gold.region_row_filter(region STRING)
RETURN
  is_account_group_member('apac_sales_team') AND region = 'APAC'
  OR is_account_group_member('metastore_admins');

In [0]:
%sql
-- Step 2: attach the function to the table as a row filter
ALTER TABLE sales_catalog.gold.customers
SET ROW FILTER sales_catalog.gold.region_row_filter ON (region);

In [0]:
%sql
-- Now a member of apac_sales_team querying this...
SELECT * FROM sales_catalog.gold.customers;
-- ...will only ever see rows where region = 'APAC', with zero changes to their query.

In [0]:
%sql
-- To remove the row filter later:
ALTER TABLE sales_catalog.gold.customers DROP ROW FILTER;

### 🎭 Column Masking

> **Column Masking** is a security feature that **hides or masks sensitive column values**
> (such as salary, PAN, Aadhaar, or credit card numbers) for unauthorized users, while
> allowing access to the remaining data.

This is the `salary → ******` example — the column still exists in the result set, but its
value is replaced for users who shouldn't see the real number. Just like row filters, a
column mask is powered by a **SQL function**.

In [0]:
%sql
-- Step 1: write a function that decides what value to show for this column
CREATE OR REPLACE FUNCTION sales_catalog.gold.mask_purchase_amount(total_purchase_usd DOUBLE)
RETURNS DOUBLE
RETURN
  CASE
    WHEN is_account_group_member('finance_admins') THEN total_purchase_usd
    ELSE NULL
  END;

In [0]:
%sql
-- Step 2: attach the function to the column as a mask
ALTER TABLE sales_catalog.gold.customers
ALTER COLUMN total_purchase_usd
SET MASK sales_catalog.gold.mask_purchase_amount;

In [0]:
%sql
-- A non-finance user querying this...
SELECT customer_name, total_purchase_usd FROM sales_catalog.gold.customers;
-- ...sees NULL in total_purchase_usd, while everything else returns normally.

In [0]:
%sql
-- To remove the column mask later:
ALTER TABLE sales_catalog.gold.customers
ALTER COLUMN total_purchase_usd DROP MASK;

### 📝 Exercise 7

Write a column mask function that fully masks a `customer_name` column for everyone except
members of `sales_admins`, showing `'REDACTED'` instead of the real name.

**Solution:**
```sql
CREATE OR REPLACE FUNCTION sales_catalog.gold.mask_customer_name(customer_name STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member('sales_admins') THEN customer_name
    ELSE 'REDACTED'
  END;

ALTER TABLE sales_catalog.gold.customers
ALTER COLUMN customer_name
SET MASK sales_catalog.gold.mask_customer_name;
```

## 📚 Summary

Let's tie everything back to our Microsoft story.

- Microsoft has huge amounts of data (HR, Finance, Sales, Marketing, AI Models, Images, PDFs).
- Without Unity Catalog, managing access for 5000+ employees across raw storage folders is
  **impossible to do safely**.
- **Unity Catalog is the brain of the data platform** — it doesn't store data, it **governs**
  it.
- Data lives in cloud storage (S3 / ADLS / GCS). **Metadata** lives in the **Unity Catalog
  Metastore**.
- The hierarchy is: **Metastore → Catalog → Schema → Table**.
- `INFORMATION_SCHEMA` lets you query that metadata using plain SQL.
- Every query is checked step-by-step: *does the table exist → who are you → do you have
  permission → row filters → column masks → where's the data → which credential → read →
  return results.*
- Beyond tables, Unity Catalog also governs **Volumes**, **External Locations**, **Storage
  Credentials**, **Permissions**, **Row Filters**, and **Column Masking**.

### 🧠 Key Takeaways to Remember

> 1. **Unity Catalog is the brain of your data platform.**
> 2. **Storage stores the data. Unity Catalog governs the data.**
> 3. **Metastore → Catalog → Schema → Table.**
> 4. **Every query passes through a governance checkpoint — every single time.**

## ✅ Knowledge Check

Try answering these before checking the solutions below — no peeking! 😉

1. Does Unity Catalog store actual data files? Why or why not?
2. What is the correct order of the Unity Catalog hierarchy?
3. What is the difference between a Metastore and `INFORMATION_SCHEMA`?
4. In the "how a query runs internally" flow, what happens right after Unity Catalog checks
   *"who are you"*?
5. Which Unity Catalog object would you use to govern access to a folder of PDF contracts?
6. What is the purpose of a Storage Credential?

### 🔓 Answers

1. **No.** Unity Catalog only stores metadata — information *about* the data (name, owner,
   location, schema, permissions). The actual data lives in cloud storage.
2. **Metastore → Catalog → Schema → Table.**
3. The **Metastore** actually *stores* the metadata. **`INFORMATION_SCHEMA`** is a read-only
   SQL interface used to *query* that stored metadata — it doesn't store anything itself.
4. Unity Catalog checks **whether you have the required permission** (e.g. `SELECT`) —
   if not, you get a permission denied error.
5. A **Volume** — Volumes are designed specifically to govern unstructured files like PDFs,
   images, and models.
6. A **Storage Credential** securely stores cloud authentication details (like an IAM Role)
   so Unity Catalog can access cloud storage on your behalf, without exposing secrets to
   users.

## 🎉 Congratulations!

You've completed the Unity Catalog KT notebook. You should now be comfortable explaining
Unity Catalog to a teammate, navigating its hierarchy, and running basic SQL/PySpark
commands against it.

**Suggested next steps:**
- Practice creating your own catalog/schema/table hierarchy in a sandbox workspace
- Explore `GRANT` / `REVOKE` statements for permissions in more depth
- Look at a real Row Filter / Column Mask example with your team's data

Happy learning! 🚀